# Lecture 12: Recurrent Networks

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 12, with material from Stanford CS231n 2026, Lecture 7.*

Every network so far takes one fixed-size input and gives one output. But lots of data comes as **sequences**: words, characters, video frames, sound. **Recurrent neural networks (RNNs)** read a sequence one step at a time, keeping a **memory** (the hidden state) as they go.

**What you'll learn**
1. The different kinds of sequence problems.
2. The **vanilla RNN**, including one we build by hand.
3. **Backpropagation through time**.
4. **Character-level language models**: predicting and generating text.
5. **Image captioning** with an RNN.
6. Why vanilla RNNs struggle with long sequences (**vanishing gradients**), and how **LSTMs** fix it.

In [ ]:
import inspect, textwrap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

## 1. Kinds of sequence problems

| Type | Example |
|---|---|
| **one to one** | image classification (everything so far) |
| **one to many** | **image captioning**: image → sequence of words |
| **many to one** | **video classification**, or the sentiment of a sentence |
| **many to many** | **translation**: a sentence → a sentence |
| **many to many, aligned** | label every frame of a video |

## 2. The vanilla RNN

At each time step, the RNN combines the new input $x_t$ with its previous hidden state $h_{t-1}$:

$$h_t = \tanh(W_{hh}\,h_{t-1} + W_{xh}\,x_t + b)$$

and can produce an output $y_t = W_{hy}\,h_t$. **The same weights are used at every time step.**

One step by hand (2 inputs, 3 hidden units):

In [ ]:
x_t = np.array([1.0, -1.0])                          # the input at this step
h_prev = np.array([0.5, 0.0, -0.5])                  # the hidden state from the previous step
W_xh = np.array([[0.2, -0.4, 0.1],
                 [0.3, 0.1, -0.2]])
W_hh = np.array([[0.5, 0.0, 0.1],
                 [0.0, 0.5, 0.0],
                 [0.2, 0.0, 0.5]])

h_new = np.tanh(x_t @ W_xh + h_prev @ W_hh)
print("from the input:   ", x_t @ W_xh)
print("from the memory:  ", h_prev @ W_hh)
print("new hidden state: ", h_new.round(3))

### An RNN we can build by hand (from CS231n 2026)
**Task:** read a sequence of 0s and 1s, and output 1 whenever the current **and** the previous input are both 1.

| x | 0 | 1 | 0 | 1 | 1 | 1 | 0 | 1 | 1 |
|---|---|---|---|---|---|---|---|---|---|
| y | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 1 |

**Q: What should the hidden state remember?** The current input and the previous input. So use a 3-number hidden state, $h = (\text{current }x,\ \text{previous }x,\ 1)$, and choose the weights so that:
* the new input goes into slot 1;
* the old slot 1 is copied into slot 2 ("previous");
* slot 3 stays 1;
* the output is $\max(\text{current} + \text{previous} - 1,\ 0)$, which is 1 only when both are 1.

(We use ReLU instead of tanh to keep it simple.)

In [ ]:
W_xh = np.array([1, 0, 0])                 # put x into slot 1
W_hh = np.array([[0, 0, 0],                # slot 1: nothing from the past
                 [1, 0, 0],                # slot 2 <- old slot 1
                 [0, 0, 1]])               # slot 3: keep the 1
W_hy = np.array([1, 1, -1])                # output: current + previous - 1

h = np.array([0, 0, 1])                    # starting hidden state
outputs = []
for x in [0, 1, 0, 1, 1, 1, 0, 1, 1]:
    h = np.maximum(0, W_hh @ h + W_xh * x)
    outputs.append(int(max(0, W_hy @ h)))
print("outputs: ", outputs)
print("expected:", [0, 0, 0, 0, 1, 1, 0, 0, 1])

It works! But in practice we can't design the weights by hand; we **learn** them.

## 3. Backpropagation through time

**Unroll** the RNN over time: $h_0 \to h_1 \to h_2 \to \dots$, with the same weights at every step. Then backprop works as usual, through the whole unrolled graph. Because the same $W$ is used at every step, **its gradient is the sum of the gradients from all the steps** (the copy rule from Lecture 6).

For one step, with $h_t = \tanh(z_t)$ and $z_t = x_tW_{xh} + h_{t-1}W_{hh} + b$, given the gradient arriving at $h_t$:
* back through tanh: $\partial L/\partial z_t = \partial L/\partial h_t \cdot (1 - h_t^2)$
* weights: $\partial L/\partial W_{xh}$ += $x_t^\top(\partial L/\partial z_t)$, and $\partial L/\partial W_{hh}$ += $h_{t-1}^\top(\partial L/\partial z_t)$
* send the gradient back to the previous step: $\partial L/\partial h_{t-1} = (\partial L/\partial z_t)\,W_{hh}^\top$

In [ ]:
def rnn_forward(params, xs, h0):
    # xs: list of inputs, one per time step, each of shape (batch, input_size). Returns [h0, h1, ..., hT].
    hs = [h0]
    for x in xs:
        hs.append(np.tanh(x @ params["Wx"] + hs[-1] @ params["Wh"] + params["b"]))
    return hs

def rnn_backward(params, xs, hs, dhs):
    # dhs[t]: gradient arriving at h_{t+1} from that step's output. Returns gradients for Wx, Wh, b.
    grads = {"Wx": np.zeros_like(params["Wx"]), "Wh": np.zeros_like(params["Wh"]), "b": np.zeros_like(params["b"])}
    dh_from_future = np.zeros_like(hs[0])
    for t in reversed(range(len(xs))):
        dh = dhs[t] + dh_from_future                  # from this step's output + from the next step
        dz = dh * (1 - hs[t + 1] ** 2)                # back through tanh
        grads["Wx"] += xs[t].T @ dz                   # the same weights at every step: ADD the gradients
        grads["Wh"] += hs[t].T @ dz
        grads["b"] += dz.sum(axis=0)
        dh_from_future = dz @ params["Wh"].T          # pass the gradient back to step t-1
    return grads

**Gradient check** (as always, before trusting it). We use a loss $L = \sum_t \sum (h_t \times u_t)$ with random $u_t$, so the gradient arriving at each $h_t$ is simply $u_t$:

In [ ]:
rng = np.random.default_rng(0)
params = {"Wx": rng.normal(0, 0.5, (4, 5)), "Wh": rng.normal(0, 0.5, (5, 5)), "b": np.zeros(5)}
xs = [rng.normal(size=(3, 4)) for _ in range(6)]       # 6 time steps, batch of 3
h0 = np.zeros((3, 5))
us = [rng.normal(size=(3, 5)) for _ in range(6)]

def test_loss():
    hs = rnn_forward(params, xs, h0)
    return sum(np.sum(h * u) for h, u in zip(hs[1:], us))

grads = rnn_backward(params, xs, rnn_forward(params, xs, h0), us)
original = params["Wh"][1, 2]
params["Wh"][1, 2] = original + 1e-6; loss_plus = test_loss()
params["Wh"][1, 2] = original - 1e-6; loss_minus = test_loss()
params["Wh"][1, 2] = original
print(f"dL/dWh[1,2]: backprop {grads['Wh'][1, 2]:.6f}, numerical {(loss_plus - loss_minus) / 2e-6:.6f}")

## 4. A character-level language model

A **language model** predicts the **next character** from the previous ones. The lecture's example: vocabulary **[h, e, l, o]**, training sequence **"hello"**:
* inputs (one per step): h, e, l, l, each as a **one-hot** vector;
* targets: the **next** character: e, l, l, o.

Notice the tricky part: after the first "l" the answer is "l", but after the second "l" it's "o". The input is the same, so **the hidden state must remember what came before**.

We add an output layer, scores $= h_tW_y + b_y$, with a softmax loss at every step. This function computes the loss and all the gradients; we'll reuse it for everything in this notebook.

In [ ]:
def language_model_loss(params, xs, targets, h0):
    # xs: list of input vectors per step; targets: list of correct next-token indices per step (-1 = ignore).
    hs = rnn_forward(params, xs, h0)
    n_counted = sum(np.sum(t >= 0) for t in targets)
    loss, dhs = 0.0, []
    grads_out = {"Wy": np.zeros_like(params["Wy"]), "by": np.zeros_like(params["by"])}
    for h, target in zip(hs[1:], targets):
        scores = h @ params["Wy"] + params["by"]
        probs = np.exp(scores - scores.max(axis=1, keepdims=True))
        probs /= probs.sum(axis=1, keepdims=True)
        counted = target >= 0                                       # skip padding positions
        loss -= np.sum(np.log(probs[counted, target[counted]]))
        d_scores = probs.copy()
        d_scores[counted, target[counted]] -= 1                     # probabilities minus one-hot
        d_scores[~counted] = 0
        d_scores /= n_counted
        grads_out["Wy"] += h.T @ d_scores
        grads_out["by"] += d_scores.sum(axis=0)
        dhs.append(d_scores @ params["Wy"].T)                       # gradient flowing into h_t
    grads = rnn_backward(params, xs, hs, dhs)
    grads.update(grads_out)
    return loss / n_counted, grads, hs[-1]

**Training helpers.** We'll train with **Adam** (Lecture 4), and **clip** the gradients so a single huge gradient can't wreck the weights (more on this in Section 7):

In [ ]:
def adam_step(params, grads, state, lr=0.01):
    state["t"] = state.get("t", 0) + 1
    for name in params:
        m = state.setdefault("m_" + name, np.zeros_like(params[name]))
        v = state.setdefault("v_" + name, np.zeros_like(params[name]))
        m[:] = 0.9 * m + 0.1 * grads[name]
        v[:] = 0.999 * v + 0.001 * grads[name] ** 2
        m_hat = m / (1 - 0.9 ** state["t"])
        v_hat = v / (1 - 0.999 ** state["t"])
        params[name] -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)

def clip_gradients(grads, max_norm=5.0):
    total = np.sqrt(sum(np.sum(g ** 2) for g in grads.values()))
    if total > max_norm:
        for name in grads:
            grads[name] *= max_norm / total

def init_rnn(input_size, hidden_size, output_size, seed=0):
    rng = np.random.default_rng(seed)
    return {"Wx": rng.normal(0, 1 / np.sqrt(input_size), (input_size, hidden_size)),
            "Wh": rng.normal(0, 1 / np.sqrt(hidden_size), (hidden_size, hidden_size)),
            "b": np.zeros(hidden_size),
            "Wy": rng.normal(0, 1 / np.sqrt(hidden_size), (hidden_size, output_size)),
            "by": np.zeros(output_size)}

Now train on "hello":

In [ ]:
vocab = ["h", "e", "l", "o"]
def one_hot(index, size):
    v = np.zeros((1, size)); v[0, index] = 1; return v

inputs = [one_hot(vocab.index(c), 4) for c in "hell"]
targets = [np.array([vocab.index(c)]) for c in "ello"]

params = init_rnn(4, 8, 4)
state = {}
for step in range(300):
    loss, grads, _ = language_model_loss(params, inputs, targets, np.zeros((1, 8)))
    adam_step(params, grads, state, lr=0.05)
print(f"loss after training: {loss:.4f}")

hs = rnn_forward(params, inputs, np.zeros((1, 8)))
for c, h in zip("hell", hs[1:]):
    prediction = vocab[int(np.argmax(h @ params["Wy"] + params["by"]))]
    print(f"after '{c}' the model predicts '{prediction}'")

**What we see:** after the two "l"s it predicts different things ("l", then "o"): the hidden state carries the context.

### Generating text: sampling
**At test time, sample one character at a time and feed it back in** as the next input. (A **temperature** below 1 makes the samples more conservative; above 1, more random.)

In [ ]:
def sample(params, first_index, length, vocab_size, rng, temperature=1.0):
    h = np.zeros((1, params["Wh"].shape[0]))
    x = one_hot(first_index, vocab_size)
    result = [first_index]
    for _ in range(length):
        h = np.tanh(x @ params["Wx"] + h @ params["Wh"] + params["b"])
        scores = (h @ params["Wy"] + params["by"]) / temperature
        probs = np.exp(scores - scores.max()); probs /= probs.sum()
        next_index = rng.choice(vocab_size, p=probs.ravel())
        result.append(next_index)
        x = one_hot(next_index, vocab_size)              # feed the sample back in
    return result

print("generated:", "".join(vocab[i] for i in sample(params, 0, 4, 4, np.random.default_rng(0))))

**Embedding layers:** multiplying a weight matrix by a one-hot vector just picks out one row of the matrix. So in practice, models skip the one-hot step and **look up** a learned vector per token directly (an "embedding layer").

## 5. Training on real text

In his blog post *The Unreasonable Effectiveness of Recurrent Neural Networks*, Andrej Karpathy trained character RNNs on Shakespeare, Wikipedia, and even the Linux source code, and they learned to produce text that looks like the real thing. Let's train on the **source code of Python's `textwrap` module** (about 20,000 characters, which comes with Python).

In [ ]:
text = inspect.getsource(textwrap)
chars = sorted(set(text))
V = len(chars)
data = np.array([chars.index(c) for c in text])
print(f"{len(text):,} characters, {V} different ones")
print(text[:150])

**Truncated backpropagation through time:** we can't backprop through all 20,000 characters at once. Instead we process **chunks of 25 characters**, **carrying the hidden state forward** from one chunk to the next, but only backpropagating within each chunk. To learn faster, we read 16 different places in the text at the same time (a batch of 16).

In [ ]:
hidden_size, chunk, batch = 128, 25, 16
code_params = init_rnn(V, hidden_size, V)
state = {}
identity = np.eye(V)
starts = np.arange(batch) * (len(data) // batch)          # 16 starting points spread through the text
position = 0
h = np.zeros((batch, hidden_size))
losses = []

for step in range(3000):
    if position + chunk + 1 >= len(data) // batch:      # reached the end: start again
        position, h = 0, np.zeros((batch, hidden_size))
    xs = [identity[data[starts + position + t]] for t in range(chunk)]      # inputs: 25 characters
    ts = [data[starts + position + t + 1] for t in range(chunk)]            # targets: the next characters
    loss, grads, h = language_model_loss(code_params, xs, ts, h)            # h carries over to the next chunk
    clip_gradients(grads)
    adam_step(code_params, grads, state, lr=3e-3)
    losses.append(loss)
    position += chunk

print(f"loss: {np.mean(losses[:50]):.2f} at the start -> {np.mean(losses[-50:]):.2f} at the end")

In [ ]:
generated = sample(code_params, chars.index("d"), 400, V, np.random.default_rng(2), temperature=0.7)
print("".join(chars[i] for i in generated))

**What we see:** it picks up how Python looks: indentation, `self.`, `def`, `=True,`, quotes. It's not working code, and some pieces are copied straight from the training text (with only 20 KB, the model partly **memorizes** it). Bigger models on more text generate much more convincing code, and today's code assistants are descendants of this idea.

Karpathy and colleagues also found **individual hidden units that track meaningful things**, without being told to: one unit turned on inside quotes, another tracked the position in the line, another the depth of nested code.

## 6. Image captioning

**Combine a CNN and an RNN:**
1. Run the image through a CNN and take its last hidden layer: a vector $v$ that describes the image.
2. Start the RNN with a special **`<START>`** token, and give it the image at every step:

$$h = \tanh(W_{xh}\,x + W_{hh}\,h + W_{ih}\,v)$$

3. Sample a word, feed it back in, and repeat until the model produces **`<END>`**.

**Our version:** "caption" an 8×8 digit with its **name**, letter by letter ("seven", then `<END>`). A handy fact: adding $W_{ih}\,v$ inside the tanh is the same as feeding the RNN the input $[x, v]$ (the character and the image stuck together), so our RNN code works unchanged.

In [ ]:
names = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
caption_vocab = ["<START>", "<END>"] + sorted(set("".join(names)))
Vc = len(caption_vocab)
max_steps = 6                                          # longest name (5 letters) + <END>

def make_sequences(digit_labels):
    # Inputs: <START> + name.  Targets: name + <END>.  Padding: -1 (ignored by the loss).
    inputs = np.zeros((len(digit_labels), max_steps), dtype=int)
    targets = np.full((len(digit_labels), max_steps), -1)
    for k, d in enumerate(digit_labels):
        letters = [caption_vocab.index(c) for c in names[d]]
        inputs[k, :len(letters) + 1] = [0] + letters
        targets[k, :len(letters) + 1] = letters + [1]
    return inputs, targets

inp, tgt = make_sequences([7])
print("inputs: ", [caption_vocab[i] for i in inp[0]])
print("targets:", [caption_vocab[i] if i >= 0 else "-" for i in tgt[0]])

In [ ]:
digits = load_digits()
X_digits = digits.data / 16.0 - 0.3
y_digits = digits.target
order = np.random.default_rng(0).permutation(len(X_digits))
train_idx, test_idx = order[:1300], order[1300:]
identity_c = np.eye(Vc)

caption_params = init_rnn(Vc + 64, 64, Vc)            # input = [one-hot letter (Vc), image (64)]
state = {}
rng = np.random.default_rng(0)
for step in range(1500):
    batch_idx = rng.choice(train_idx, 64)
    images_batch = X_digits[batch_idx]
    inp, tgt = make_sequences(y_digits[batch_idx])
    xs = [np.hstack([identity_c[inp[:, t]], images_batch]) for t in range(max_steps)]
    ts = [tgt[:, t] for t in range(max_steps)]
    loss, grads, _ = language_model_loss(caption_params, xs, ts, np.zeros((64, 64)))
    clip_gradients(grads)
    adam_step(caption_params, grads, state, lr=5e-3)
print(f"final training loss: {loss:.3f}")

**Captioning a test image:** start with `<START>`, take the most likely next letter, feed it back in, stop at `<END>`.

In [ ]:
def caption(image):
    h = np.zeros((1, 64))
    letter = 0                                                     # <START>
    result = ""
    for _ in range(8):
        x = np.hstack([identity_c[[letter]], image[None]])
        h = np.tanh(x @ caption_params["Wx"] + h @ caption_params["Wh"] + caption_params["b"])
        letter = int(np.argmax(h @ caption_params["Wy"] + caption_params["by"]))
        if letter == 1:                                            # <END>
            break
        result += caption_vocab[letter]
    return result

captions = [caption(X_digits[i]) for i in test_idx]
correct = [c == names[y_digits[i]] for c, i in zip(captions, test_idx)]
print(f"test images captioned exactly right: {np.mean(correct):.1%}")
print("examples:", [(names[y_digits[i]], c) for c, i in list(zip(captions, test_idx))[:6]])
mistakes = [c for c, i in zip(captions, test_idx) if c != names[y_digits[i]]]
print(f"of the {len(mistakes)} mistakes, {sum(c in names for c in mistakes)} are correctly spelled names of the wrong digit")

The RNN writes the digit's name letter by letter and decides by itself when to stop. Its mistakes are almost all **recognition** mistakes (the right spelling of the wrong digit), like real captioning systems that call a fur coat "a cat". The same idea with real CNN features and a word vocabulary produced the 2015 captioning results ("A cat sitting on a suitcase on the floor").

## 7. The problem: vanishing and exploding gradients

Backprop from $h_t$ to $h_{t-1}$ multiplies the gradient by $W_{hh}^\top$ (and by tanh's slope). Going back $T$ steps multiplies by $W_{hh}^\top$ **$T$ times**. So:
* if $W_{hh}$ makes vectors **bigger** (largest singular value > 1), gradients can **explode**;
* if it makes them **smaller** (< 1), gradients **vanish**.

To see it cleanly, use $W_{hh} = s\times$(a rotation), which scales every vector by exactly $s$:

In [ ]:
H = 64
rotation, _ = np.linalg.qr(np.random.default_rng(0).normal(size=(H, H)))   # a random rotation matrix

for s in [0.9, 1.0, 1.1]:
    W = s * rotation
    gradient = np.ones(H) / np.sqrt(H)            # a gradient of size 1 at step 50
    for step in range(50):
        gradient = gradient @ W.T                 # back one step (ignoring tanh)
    print(f"s = {s}: gradient size after going back 50 steps = {np.linalg.norm(gradient):.3e}   (s^50 = {s**50:.3e})")

**Exponential** shrinking or growing. (tanh's slope is below 1, which makes vanishing even more likely.)
* **Exploding gradients → gradient clipping:** if the gradient is too big, scale it down (we've been doing this with `clip_gradients`).
* **Vanishing gradients → change the architecture:** the LSTM.

## 8. Long Short-Term Memory (LSTM)

The LSTM (Hochreiter & Schmidhuber, 1997) keeps **two** states: a **cell state** $c_t$ (long-term memory) and a hidden state $h_t$. At each step it computes four **gates** from $h_{t-1}$ and $x_t$:
* **i** (input gate): how much to write to the cell;
* **f** (forget gate): how much of the old cell to keep;
* **o** (output gate): how much of the cell to reveal;
* **g**: what to write (a value between −1 and 1).

$$c_t = f\odot c_{t-1} + i\odot g, \qquad h_t = o\odot\tanh(c_t)$$

**The key:** going back from $c_t$ to $c_{t-1}$ is just **multiplication by $f$**, with no matrix multiply. If $f\approx1$, the gradient flows back through the cell states almost unchanged, like a ResNet shortcut. And with $f = 1$, $i = 0$, the cell **remembers its value forever**:

In [ ]:
c = np.array([0.8, -0.3])                     # something stored in the cell
for step in range(100):
    f, i, g = 1.0, 0.0, np.random.default_rng(step).uniform(-1, 1, 2)    # forget gate open, input gate closed
    c = f * c + i * g
print("cell after 100 steps:", c, "  (unchanged)")

The LSTM's forward and backward pass for one sequence (the backward pass follows the same upstream × local rule; the line `dc_next = dc * f` is the "highway"):

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def lstm_forward(params, xs, h0):
    H = h0.shape[1]
    h, c = h0, np.zeros_like(h0)
    steps = []
    for x in xs:
        a = x @ params["Wx"] + h @ params["Wh"] + params["b"]          # all four gates at once
        i, f, o = sigmoid(a[:, :H]), sigmoid(a[:, H:2*H]), sigmoid(a[:, 2*H:3*H])
        g = np.tanh(a[:, 3*H:])
        c_new = f * c + i * g
        h_new = o * np.tanh(c_new)
        steps.append((x, h, c, i, f, o, g, c_new))
        h, c = h_new, c_new
    return h, steps

def lstm_backward(params, steps, dh_last):
    # Gradients when only the LAST hidden state has a loss attached (many-to-one).
    H = dh_last.shape[1]
    grads = {"Wx": np.zeros_like(params["Wx"]), "Wh": np.zeros_like(params["Wh"]), "b": np.zeros_like(params["b"])}
    dh, dc_next = dh_last, np.zeros_like(dh_last)
    for x, h_prev, c_prev, i, f, o, g, c in reversed(steps):
        do = dh * np.tanh(c)
        dc = dc_next + dh * o * (1 - np.tanh(c) ** 2)
        di, df, dg = dc * g, dc * c_prev, dc * i
        dc_next = dc * f                                               # the cell-state highway: just "times f"
        da = np.hstack([di * i * (1 - i), df * f * (1 - f), do * o * (1 - o), dg * (1 - g ** 2)])
        grads["Wx"] += x.T @ da
        grads["Wh"] += h_prev.T @ da
        grads["b"] += da.sum(axis=0)
        dh = da @ params["Wh"].T
    return grads

### A memory race: vanilla RNN vs. LSTM
**Task:** the first step of each sequence is a **signal** (0 or 1); then come $T - 1$ steps of **random noise**. At the end, the network must output the signal. It has to carry one bit across $T$ steps while the noise pushes its state around.

In [ ]:
def memory_task(n, T, rng):
    signal = rng.integers(0, 2, n)
    first = np.zeros((n, 4)); first[np.arange(n), signal] = 1         # one-hot signal in input slot 0 or 1
    xs = [first]
    for t in range(T - 1):
        noise = np.zeros((n, 4)); noise[:, 2:] = rng.normal(0, 1, (n, 2))   # noise in slots 2-3
        xs.append(noise)
    return xs, signal

One training function for both kinds of network. The only difference is which forward/backward functions it calls. For the LSTM, we try two starting values for the **forget gate's bias**: 1 (so $f\approx0.73$ at the start) and 3 ($f\approx0.95$).

In [ ]:
def train_memory(cell, T, forget_bias=1.0, steps=800, H=32, seed=0):
    rng = np.random.default_rng(seed)
    n_gates = 4 if cell == "lstm" else 1
    params = {"Wx": rng.normal(0, 0.5, (4, n_gates * H)), "Wh": rng.normal(0, 1 / np.sqrt(H), (H, n_gates * H)),
              "b": np.zeros(n_gates * H), "Wy": rng.normal(0, 1 / np.sqrt(H), (H, 2)), "by": np.zeros(2)}
    if cell == "lstm":
        params["b"][H:2 * H] = forget_bias
    state = {}
    for step in range(steps + 1):
        xs, y = memory_task(64 if step < steps else 2000, T, rng if step < steps else np.random.default_rng(999))
        if cell == "rnn":
            hs = rnn_forward(params, xs, np.zeros((len(y), H))); h_last = hs[-1]
        else:
            h_last, lstm_steps = lstm_forward(params, xs, np.zeros((len(y), H)))
        scores = h_last @ params["Wy"] + params["by"]
        if step == steps:                                   # the last round is a test on fresh sequences
            return np.mean(np.argmax(scores, axis=1) == y)
        probs = np.exp(scores - scores.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
        probs[np.arange(len(y)), y] -= 1; d_scores = probs / len(y)
        dh_last = d_scores @ params["Wy"].T
        if cell == "rnn":
            dhs = [np.zeros_like(h_last)] * (T - 1) + [dh_last]
            grads = rnn_backward(params, xs, hs, dhs)
        else:
            grads = lstm_backward(params, lstm_steps, dh_last)
        grads["Wy"] = h_last.T @ d_scores; grads["by"] = d_scores.sum(axis=0)
        clip_gradients(grads)
        adam_step(params, grads, state, lr=1e-2)

In [ ]:
# Sequences of 100 steps, 3 random seeds per model (this cell takes a minute or two).
print("test accuracy on 100-step sequences (50% = guessing), for 3 different random seeds:")
for name, cell, bias in [("vanilla RNN", "rnn", None), ("LSTM, forget bias 1", "lstm", 1.0), ("LSTM, forget bias 3", "lstm", 3.0)]:
    results = [train_memory(cell, 100, forget_bias=bias if bias else 1.0, seed=seed) for seed in range(3)]
    print(f"  {name:20s}: " + ", ".join(f"{r:.0%}" for r in results))

**What we see** (with different random seeds, the exact numbers can vary):
* **The vanilla RNN is unreliable:** depending on the random seed, it sometimes learns the task and sometimes is stuck guessing. (Adam and gradient clipping partly compensate for its tiny gradients, so on a task this simple it can get lucky.)
* **The LSTM with forget bias 1 fails every time.** At the start $f\approx0.73$, and $0.73^{100}$ is tiny, so its memory fades just like a vanilla RNN's.
* **The LSTM with forget bias 3 succeeds every time.** With $f\approx0.95$ at the start, the cell carries the signal (and its gradient) across all 100 steps. That's the "highway" idea: it works **when $f$ is close to 1**, which is why people initialize the forget-gate bias to a positive value.

**Do LSTMs solve vanishing gradients?** Not completely, but they make it **much easier** to learn long-range dependencies. That's why, for years, LSTMs were the standard for language, speech, and translation.

## 9. Other RNNs

* **GRU** (2014): a simpler gated RNN, with two gates and no separate cell state; works about as well as an LSTM.
* **Multilayer RNNs:** stack RNN layers, where one layer's hidden states are the next layer's inputs.
* **Modern RNNs / state-space models** (S4, Mamba, RWKV; CS231n 2026): keep the hidden state, but make the recurrence **linear**, $h_t = a\,h_{t-1} + b\,x_t$. Then the whole sequence can be computed in parallel, and memory length is controlled by how close $a$ is to 1, just like the forget gate.

**RNN pros and cons:**
* **Pros:** handle any sequence length; the model size doesn't grow with the length; in principle, they can use information from far back.
* **Cons:** slow (each step waits for the previous one); in practice, it's hard to use information from far back.

These cons are what **attention** and **Transformers** fix (Lecture 13).

## 10. In PyTorch (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch
import torch, torch.nn as nn
x = torch.randn(8, 25, 16)                                 # (batch, time steps, features)
for cls in [nn.RNN, nn.LSTM, nn.GRU]:
    layer = cls(input_size=16, hidden_size=32, batch_first=True)
    output, _ = layer(x)
    print(f"{cls.__name__}: output shape {tuple(output.shape)}, parameters {sum(p.numel() for p in layer.parameters())}")
# Gradient clipping in PyTorch, after loss.backward():
#     torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

## Summary
* **RNNs** process sequences step by step with a hidden state: $h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t)$, the same weights at every step.
* **Backprop through time:** unroll and backprop; shared weights → add up the gradients. **Truncated** BPTT for long sequences.
* **Language models** predict the next token; **sample** and feed back to generate text.
* **Image captioning:** CNN features condition the RNN at every step; `<START>` … `<END>`.
* **Vanishing/exploding gradients:** repeated multiplication by $W_{hh}$. **Clip** to stop explosions; use an **LSTM** against vanishing.
* **LSTM:** gates i, f, o, g; the cell state $c_t = f\odot c_{t-1} + i\odot g$ is a gradient highway when $f\approx1$.

### Check your understanding
1. In the "hello" model, why must the hidden state remember the past?
2. Why do we **add** the gradients of $W_{hh}$ over time steps?
3. What does gradient clipping do, and what problem does it solve?
4. Why can an LSTM remember things longer than a vanilla RNN?

<details><summary>Answers</summary>

1. The input "l" appears twice with different correct next characters; only the memory of what came before tells them apart.
2. The same $W_{hh}$ is used at every step (the copy rule).
3. It scales the gradient down when it's too big, keeping its direction; it prevents exploding gradients from wrecking the weights.
4. Its cell state is updated by $c_t = f\odot c_{t-1} + \dots$, so with $f\approx1$ both the information and its gradient pass through many steps almost unchanged.
</details>

**Next:** Lecture 13, attention.